In [1]:
# ==========================================
# YikYak Sentiment Model Testing Script
# ==========================================

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from detoxify import Detoxify

# ==========================================
# 1️⃣ Load Model + Tokenizer
# ==========================================
model_path = "yikyak_sentiment_model_finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Running on:", device)

# ==========================================
# 2️⃣ Load CSV Dataset
# ==========================================
df = pd.read_csv("posts_test.csv", sep=";")

# clean dataset
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

texts = df["text"].tolist()

print(f"Loaded {len(texts)} rows")

# ==========================================
# 3️⃣ Tokenize + Predict (BATCHED)
# ==========================================
batch_size = 16
predictions = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i + batch_size]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)

    predictions.extend(preds.cpu().numpy())

# attach sentiment predictions
df["prediction"] = predictions

# ==========================================
# 4️⃣ Optional Accuracy Check
# ==========================================
if "label" in df.columns:
    acc = accuracy_score(df["label"], df["prediction"])
    print("Accuracy:", acc)

# ==========================================
# 5️⃣ PROFANITY DETECTION (ADDED SECTION)
# ==========================================
print("\nRunning profanity detection...")

profanity_model = Detoxify("original")

def profanity_label(text):
    scores = profanity_model.predict(text)

    # 🔴 Obscene (slurs / identity attacks)
    if scores["identity_attack"] > 0.6:
        return "🔴"

    # 🟡 Inappropriate (general profanity)
    elif scores["obscene"] > 0.5 or scores["toxicity"] > 0.7:
        return "🟡"

    # 🟢 Clean
    else:
        return "🟢"

# apply profanity labels
profanity_labels = []
for text in tqdm(df["text"], desc="Profanity pass"):
    profanity_labels.append(profanity_label(text))

df["profanity_emoji"] = profanity_labels

# ==========================================
# 6️⃣ Save Results
# ==========================================
output_file = "predicted_output1.csv"
df.to_csv(output_file, index=False)

print("\n✅ Finished!")
print("Saved to:", output_file)
print(df.head())

/Users/jacobhopkins/PycharmProjects/YikHak'd/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on: cpu
Loaded 991 rows


 47%|████▋     | 29/62 [00:45<00:52,  1.58s/it]


KeyboardInterrupt: 